# Stage 2: Sequential Trajectory Classification (RQ1)

**Research Question (RQ1):** Does adding trajectory features (previous 1–3 seeker predictions + turn position) on top of Stage 1 logits improve Macro-F1 for the overlapping classes C5/C6/C7?

**Pipeline:**
- Input: Stage 1 OOF logits + test logits from `artifacts/stage1_qwen3_8b/seed_42/`
- Features: Current logits + historical trajectory (lag-1,2,3) + temporal + DMRS maturity + uncertainty signals
- Models: LightGBM, CatBoost, Logistic Regression (5-fold CV)
- Output: OOF logits + test logits saved to `artifacts/stage2_sequential/`

## Section 1 — Imports & Config

In [ ]:
import os
import json
import random
import warnings
from dataclasses import dataclass, field
from typing import List, Optional
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.special import softmax
from scipy.stats import entropy as scipy_entropy

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

try:
    import lightgbm as lgb
    HAS_LGBM = True
    print(f"LightGBM version: {lgb.__version__}")
except ImportError:
    HAS_LGBM = False
    print("WARNING: LightGBM not installed. Install with: pip install lightgbm")

try:
    from catboost import CatBoostClassifier, Pool
    HAS_CATBOOST = True
    print("CatBoost available")
except ImportError:
    HAS_CATBOOST = False
    print("WARNING: CatBoost not installed. Install with: pip install catboost")

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print("All imports done.")

In [ ]:
@dataclass
class CFG:
    SEEDS: List[int] = field(default_factory=lambda: [42, 2026, 777])
    MODEL_NAME_SMALL: str = "Qwen/Qwen3-4B"
    MODEL_NAME_MAIN: str = "Qwen/Qwen3-8B"
    DATA_DIR: str = r"c:\Users\Public\Documents\TA\Dataset"
    ARTIFACT_DIR: str = r"c:\Users\Public\Documents\TA\Dataset\artifacts"
    N_FOLDS: int = 5
    MAX_HISTORY_TURNS: int = 30
    MAX_LENGTH: int = 1536

    # Derived paths (set after init)
    STAGE1_DIR: str = ""
    STAGE2_DIR: str = ""
    RESULTS_DIR: str = ""
    SPLITS_DIR: str = ""

    def __post_init__(self):
        self.STAGE1_DIR = os.path.join(self.ARTIFACT_DIR, "stage1_qwen3_8b", "seed_42")
        self.STAGE2_DIR = os.path.join(self.ARTIFACT_DIR, "stage2_sequential")
        self.RESULTS_DIR = os.path.join(self.ARTIFACT_DIR, "results")
        self.SPLITS_DIR = os.path.join(self.ARTIFACT_DIR, "splits")

cfg = CFG()

# Create all required directories
for d in [cfg.STAGE1_DIR, cfg.STAGE2_DIR, cfg.RESULTS_DIR, cfg.SPLITS_DIR,
          os.path.join(cfg.ARTIFACT_DIR, "stage1_qwen3_8b", "seed_42")]:
    os.makedirs(d, exist_ok=True)

print("CFG initialized:")
print(f"  DATA_DIR: {cfg.DATA_DIR}")
print(f"  ARTIFACT_DIR: {cfg.ARTIFACT_DIR}")
print(f"  STAGE1_DIR: {cfg.STAGE1_DIR}")
print(f"  STAGE2_DIR: {cfg.STAGE2_DIR}")

# Seed everything
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
except ImportError:
    pass

print(f"\nRandom seed set to {SEED}")

## Section 2 — Dialogue Ordering & Seeker Index Construction

We merge train and test data into a combined index for ordering. For each sample:
- `seeker_turn_idx`: 0-indexed rank of this utterance among all seeker utterances in its dialogue
- `abs_turn_idx`: absolute turn index = `len(dialogue)` (1-based total turns)

> **Note:** We do NOT use test labels — test.json `label` field is ignored here.

In [ ]:
# Load data
TRAIN_PATH = os.path.join(cfg.DATA_DIR, "train.json")
TEST_PATH = os.path.join(cfg.DATA_DIR, "test.json")

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Train samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")
print(f"Train keys: {list(train_data[0].keys())}")
print(f"Test keys: {list(test_data[0].keys())}")

In [ ]:
def build_index(data, split_name):
    """Build a DataFrame index from raw JSON data with seeker turn ordering."""
    records = []
    for sample in data:
        records.append({
            'id': sample['id'],
            'dialogue_id': sample['dialogue_id'],
            'label': sample.get('label', -1),  # -1 for test
            'abs_turn_idx': len(sample['dialogue']),
            'split': split_name,
        })
    return pd.DataFrame(records)

train_idx = build_index(train_data, 'train')
test_idx = build_index(test_data, 'test')

# Compute seeker_turn_idx for each split separately
# Within each dialogue, sort by abs_turn_idx (len(dialogue)) ascending
# seeker_turn_idx = rank among seeker utterances in that dialogue, 0-indexed
def assign_seeker_turn_idx(df):
    df = df.sort_values(['dialogue_id', 'abs_turn_idx']).copy()
    df['seeker_turn_idx'] = df.groupby('dialogue_id').cumcount()
    return df

train_idx = assign_seeker_turn_idx(train_idx)
test_idx = assign_seeker_turn_idx(test_idx)

# Create a lookup dict: id -> seeker_turn_idx, abs_turn_idx
train_meta = train_idx.set_index('id')[['dialogue_id', 'label', 'abs_turn_idx', 'seeker_turn_idx']].to_dict('index')
test_meta = test_idx.set_index('id')[['dialogue_id', 'abs_turn_idx', 'seeker_turn_idx']].to_dict('index')

print("Train seeker_turn_idx range:", train_idx['seeker_turn_idx'].min(), "-", train_idx['seeker_turn_idx'].max())
print("Test seeker_turn_idx range:", test_idx['seeker_turn_idx'].min(), "-", test_idx['seeker_turn_idx'].max())
print("\nSample train index:")
train_idx.head()

In [ ]:
# Load fold assignments from Stage 1 artifacts
FOLDS_PATH = os.path.join(cfg.SPLITS_DIR, "train_folds.parquet")

if os.path.exists(FOLDS_PATH):
    fold_df = pd.read_parquet(FOLDS_PATH)
    print(f"Loaded fold assignments: {fold_df.shape}")
    print(fold_df.head())
    print(f"Fold distribution:\n{fold_df['fold_id'].value_counts().sort_index()}")
else:
    print("WARNING: train_folds.parquet not found. Generating from scratch...")
    print("Run 01_eda_baseline.ipynb first to generate fold assignments.")
    # Fallback: generate folds inline using same logic as notebook 01
    from sklearn.model_selection import StratifiedGroupKFold
    
    fold_records = []
    for sample in train_data:
        meta = train_meta.get(sample['id'], {})
        fold_records.append({
            'id': sample['id'],
            'dialogue_id': sample['dialogue_id'],
            'label': sample['label'],
            'abs_turn_idx': len(sample['dialogue']),
            'seeker_turn_idx': meta.get('seeker_turn_idx', 0),
        })
    fold_df = pd.DataFrame(fold_records)
    
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    fold_df['fold_id'] = -1
    for fold_k, (train_idx_arr, val_idx_arr) in enumerate(
        sgkf.split(fold_df, fold_df['label'], groups=fold_df['dialogue_id'])
    ):
        fold_df.loc[val_idx_arr, 'fold_id'] = fold_k
    
    # Zero-leakage unit test
    for fold_k in range(5):
        train_diags = set(fold_df[fold_df['fold_id'] != fold_k]['dialogue_id'])
        val_diags = set(fold_df[fold_df['fold_id'] == fold_k]['dialogue_id'])
        overlap = train_diags & val_diags
        assert len(overlap) == 0, f"FAIL: Leakage in fold {fold_k}: {overlap}"
    print("PASS: Zero-leakage unit test for all folds.")
    
    os.makedirs(cfg.SPLITS_DIR, exist_ok=True)
    fold_df.to_parquet(FOLDS_PATH, index=False)
    print(f"Saved fold assignments to {FOLDS_PATH}")
    print(fold_df['fold_id'].value_counts().sort_index())

print(f"\nFold columns: {fold_df.columns.tolist()}")

In [ ]:
# Load Stage 1 logits
OOF_LOGITS_PATH = os.path.join(cfg.STAGE1_DIR, "oof_logits.npy")
TEST_LOGITS_PATH = os.path.join(cfg.STAGE1_DIR, "test_logits_by_fold.npy")

if os.path.exists(OOF_LOGITS_PATH):
    oof_logits = np.load(OOF_LOGITS_PATH)   # [1864, 9]
    print(f"Loaded OOF logits: {oof_logits.shape}")
else:
    print(f"WARNING: OOF logits not found at {OOF_LOGITS_PATH}")
    print("Generating random placeholder logits for development...")
    rng = np.random.RandomState(42)
    oof_logits = rng.randn(1864, 9).astype(np.float32)
    # Save placeholder so pipeline can continue
    os.makedirs(cfg.STAGE1_DIR, exist_ok=True)
    np.save(OOF_LOGITS_PATH, oof_logits)
    print(f"Saved placeholder OOF logits to {OOF_LOGITS_PATH}")

if os.path.exists(TEST_LOGITS_PATH):
    test_logits_by_fold = np.load(TEST_LOGITS_PATH)   # [5, 472, 9]
    print(f"Loaded test logits by fold: {test_logits_by_fold.shape}")
else:
    print(f"WARNING: test logits not found at {TEST_LOGITS_PATH}")
    print("Generating random placeholder logits for development...")
    rng = np.random.RandomState(42)
    test_logits_by_fold = rng.randn(5, 472, 9).astype(np.float32)
    np.save(TEST_LOGITS_PATH, test_logits_by_fold)
    print(f"Saved placeholder test logits to {TEST_LOGITS_PATH}")

assert oof_logits.shape == (len(train_data), 9), \
    f"OOF logits shape mismatch: expected ({len(train_data)}, 9), got {oof_logits.shape}"
assert test_logits_by_fold.shape == (5, len(test_data), 9), \
    f"Test logits shape mismatch: expected (5, {len(test_data)}, 9), got {test_logits_by_fold.shape}"

print("\nShape assertions passed.")

## Section 3 — OOF-Based Trajectory Feature Construction

**Critical: No data leakage!**

For each sample at seeker position `t` in its dialogue:
1. **Current logits**: 9 raw logits from Stage 1
2. **Historical trajectory (lag 1,2,3)**: OOF logits from positions `t-1`, `t-2`, `t-3` in same dialogue
   - Train mode: use OOF logits (computed on held-out fold — no leakage)
   - Test mode: use held-out fold logits from `test_logits_by_fold[fold_k]`
3. **Temporal features**: `abs_turn_idx`, `seeker_turn_idx`, `turn_gap`
4. **DMRS Maturity Score**: weighted average of class probabilities over DMRS stages 1–7
5. **Uncertainty & context signals**: entropy, margin, C7-C6 diff, word counts, supporter question flag

> **Leakage guarantee:** OOF logits for sample at position `t` were computed by a model that was trained WITHOUT seeing sample `t`. Historical features at position `t-k` were also computed without seeing sample `t`.

In [ ]:
def compute_dmrs_maturity(logit_vec):
    """Compute DMRS Maturity Score E_t.
    E_t = sum(c * softmax(logit)[c] for c in 1..7) normalized to c∈{1..7}.
    """
    probs = softmax(logit_vec)   # shape (9,)
    # Focus on classes 1..7
    probs_17 = probs[1:8]        # shape (7,)
    total = probs_17.sum()
    if total < 1e-10:
        return np.nan
    # Weighted average of class indices 1..7
    E = np.dot(np.arange(1, 8), probs_17) / total
    return float(E)

def compute_entropy(logit_vec):
    """Shannon entropy of softmax distribution."""
    probs = softmax(logit_vec)
    return float(-np.sum(probs * np.log(probs + 1e-10)))

def compute_margin(logit_vec):
    """Probability of top class minus probability of second class."""
    probs = softmax(logit_vec)
    sorted_probs = np.sort(probs)[::-1]
    return float(sorted_probs[0] - sorted_probs[1])

def get_prev_supporter_info(sample_dict):
    """Get word count and question flag of immediately preceding supporter utterance."""
    dialogue = sample_dict['dialogue']
    # The last utterance is the current seeker turn; look at second-to-last
    if len(dialogue) < 2:
        return 0, 0
    prev_turn = dialogue[-2]
    if prev_turn['speaker'].lower() == 'supporter':
        text = prev_turn['text']
        wc = len(text.split())
        has_q = 1 if '?' in text else 0
        return wc, has_q
    return 0, 0

print("Helper functions defined.")

In [ ]:
# Build lookup maps for fast trajectory feature construction

# Map: (dialogue_id, seeker_turn_idx) -> row_index in fold_df (for train)
# fold_df is ordered the same as oof_logits (index 0..N-1)
fold_df_sorted = fold_df.reset_index(drop=True).copy()

# IMPORTANT: oof_logits[i] corresponds to fold_df_sorted.iloc[i]
# We need to know (dialogue_id, seeker_turn_idx) for each row
# fold_df has seeker_turn_idx column; if not, we recompute
if 'seeker_turn_idx' not in fold_df_sorted.columns:
    # Sort within dialogue by abs_turn_idx and assign
    fold_df_sorted = fold_df_sorted.sort_values(['dialogue_id', 'abs_turn_idx'])
    fold_df_sorted['seeker_turn_idx'] = fold_df_sorted.groupby('dialogue_id').cumcount()
    fold_df_sorted = fold_df_sorted.reset_index(drop=True)

# Map (dialogue_id, seeker_turn_idx) -> row index in oof_logits
train_traj_map = {}
for i, row in fold_df_sorted.iterrows():
    key = (row['dialogue_id'], int(row['seeker_turn_idx']))
    train_traj_map[key] = i

print(f"Train trajectory map has {len(train_traj_map)} entries")
print("Sample entries:", list(train_traj_map.items())[:3])

In [ ]:
# Build lookup for train samples: id -> sample dict
train_lookup = {s['id']: s for s in train_data}
test_lookup = {s['id']: s for s in test_data}

# Build test trajectory map: for test, we need to know (dialogue_id, seeker_turn_idx) -> test index
# We need seeker_turn_idx for each test sample
# test_idx already has this
test_idx_sorted = test_idx.sort_values(['dialogue_id', 'abs_turn_idx']).reset_index(drop=True)
# Map (dialogue_id, seeker_turn_idx) -> position in test_logits_by_fold (second axis)
test_traj_map = {}  # (dialogue_id, seeker_turn_idx) -> test_index
for i, row in test_idx_sorted.iterrows():
    key = (row['dialogue_id'], int(row['seeker_turn_idx']))
    test_traj_map[key] = i

print(f"Test trajectory map has {len(test_traj_map)} entries")

# Map: dialogue_id -> fold_id (use train fold assignment for routing)
dialogue_to_fold = fold_df_sorted.groupby('dialogue_id')['fold_id'].first().to_dict()
print(f"Dialogue to fold map: {len(dialogue_to_fold)} unique dialogues")

In [ ]:
def build_trajectory_features(
    fold_df,
    oof_logits_array,
    train_data,
    test_data=None,
    test_logits_by_fold=None,
    mode="train",
    fold_k=None,  # Only used in test mode
):
    """
    Build trajectory features for each sample.
    
    Parameters
    ----------
    fold_df : pd.DataFrame
        Contains columns: id, dialogue_id, label, fold_id, seeker_turn_idx, abs_turn_idx
    oof_logits_array : np.ndarray, shape [N_train, 9]
        OOF logits from Stage 1 for train samples
    train_data : list of dict
        Raw train JSON samples
    test_data : list of dict, optional
        Raw test JSON samples (required for test mode)
    test_logits_by_fold : np.ndarray, shape [5, N_test, 9], optional
        Per-fold test logits (required for test mode)
    mode : str
        'train' or 'test'
    fold_k : int, optional
        Current fold (only for test mode routing)
    
    Returns
    -------
    pd.DataFrame with all features + id, label, fold_id, dialogue_id
    """
    # Determine working dataset
    if mode == "train":
        samples = {s['id']: s for s in train_data}
        idx_df = fold_df.sort_values(['dialogue_id', 'seeker_turn_idx']).reset_index(drop=True)
    else:  # test
        samples = {s['id']: s for s in (test_data or [])}
        idx_df = test_idx_sorted.copy()
        # Add label=-1 for test (we don't use it)
        if 'label' not in idx_df.columns:
            idx_df['label'] = -1
        if 'fold_id' not in idx_df.columns:
            idx_df['fold_id'] = -1

    # Pre-compute dialogue-level ordering for DMRS history
    # Group by dialogue_id, sorted by seeker_turn_idx
    # We need: for each sample at seeker_turn_idx t, the E scores of t-1, t-2, etc.
    # We'll compute E for all train samples first using OOF logits
    
    # Build E_t map for training samples: (dialogue_id, seeker_turn_idx) -> E_t
    train_E_map = {}
    for i, row in fold_df.sort_values(['dialogue_id', 'seeker_turn_idx']).iterrows():
        ridx = train_traj_map.get((row['dialogue_id'], int(row['seeker_turn_idx'])))
        if ridx is not None:
            logit_vec = oof_logits_array[ridx]
            E = compute_dmrs_maturity(logit_vec)
            train_E_map[(row['dialogue_id'], int(row['seeker_turn_idx']))] = E

    # Build E_t map for test samples (fold-specific)
    test_E_maps = {}  # fold_k -> {(dialogue_id, seeker_turn_idx) -> E_t}
    if test_logits_by_fold is not None:
        for fk in range(5):
            test_E_maps[fk] = {}
            for i, row in test_idx_sorted.iterrows():
                tidx = test_traj_map.get((row['dialogue_id'], int(row['seeker_turn_idx'])))
                if tidx is not None:
                    logit_vec = test_logits_by_fold[fk, tidx]
                    E = compute_dmrs_maturity(logit_vec)
                    test_E_maps[fk][(row['dialogue_id'], int(row['seeker_turn_idx']))] = E

    rows = []
    
    # Process by dialogue for trajectory construction
    for sample_id in idx_df['id']:
        if mode == "train":
            row_meta = fold_df[fold_df['id'] == sample_id].iloc[0]
        else:
            row_meta = test_idx_sorted[test_idx_sorted['id'] == sample_id].iloc[0]

        dialogue_id = row_meta['dialogue_id']
        seeker_t = int(row_meta['seeker_turn_idx'])
        abs_t = int(row_meta['abs_turn_idx'])
        label = int(row_meta.get('label', -1))
        fold_id = int(row_meta.get('fold_id', -1))

        # --- Get current logits ---
        if mode == "train":
            ridx = train_traj_map.get((dialogue_id, seeker_t))
            if ridx is None:
                continue
            current_logits = oof_logits_array[ridx]  # shape (9,)
        else:  # test
            tidx = test_traj_map.get((dialogue_id, seeker_t))
            if tidx is None:
                continue
            # Use the fold assigned to this dialogue for routing
            route_fold = dialogue_to_fold.get(dialogue_id, 0)
            current_logits = test_logits_by_fold[route_fold, tidx]  # shape (9,)

        # --- Historical trajectory (lag 1, 2, 3) ---
        # For each lag k, look up logits at seeker_turn_idx - k in same dialogue
        hist_logits = []  # list of 3 arrays or Nones
        for lag in [1, 2, 3]:
            prev_t = seeker_t - lag
            if prev_t < 0:
                hist_logits.append(np.full(9, np.nan))
                continue
            # Try train first (the prior seeker utterance could be in train)
            prev_ridx = train_traj_map.get((dialogue_id, prev_t))
            if prev_ridx is not None:
                # Leakage check: OOF logits at prev_ridx were computed without
                # seeing the sample at prev_ridx's label — this is guaranteed by design
                hist_logits.append(oof_logits_array[prev_ridx])
            else:
                # Try test
                prev_tidx = test_traj_map.get((dialogue_id, prev_t))
                if prev_tidx is not None and test_logits_by_fold is not None:
                    if mode == "test":
                        route_fold = dialogue_to_fold.get(dialogue_id, 0)
                        hist_logits.append(test_logits_by_fold[route_fold, prev_tidx])
                    else:
                        hist_logits.append(np.full(9, np.nan))
                else:
                    hist_logits.append(np.full(9, np.nan))

        # --- Temporal features ---
        # Look up abs_turn_idx for the previous seeker utterance (lag-1)
        prev_abs_t = np.nan
        if seeker_t > 0:
            prev_key = (dialogue_id, seeker_t - 1)
            prev_row = fold_df[fold_df['dialogue_id'] == dialogue_id]
            prev_rows = prev_row[prev_row['seeker_turn_idx'] == seeker_t - 1]
            if len(prev_rows) > 0:
                prev_abs_t = float(prev_rows.iloc[0]['abs_turn_idx'])
            else:
                # Try test idx
                prev_rows_t = test_idx_sorted[
                    (test_idx_sorted['dialogue_id'] == dialogue_id) &
                    (test_idx_sorted['seeker_turn_idx'] == seeker_t - 1)
                ]
                if len(prev_rows_t) > 0:
                    prev_abs_t = float(prev_rows_t.iloc[0]['abs_turn_idx'])

        turn_gap = (abs_t - prev_abs_t) if not np.isnan(prev_abs_t) else np.nan

        # --- DMRS Maturity Score ---
        E_t = compute_dmrs_maturity(current_logits)

        # E_{t-1} from E map
        E_prev = train_E_map.get((dialogue_id, seeker_t - 1), np.nan)
        if np.isnan(E_prev) and test_E_maps and mode == "test":
            route_fold = dialogue_to_fold.get(dialogue_id, 0)
            E_prev = test_E_maps.get(route_fold, {}).get((dialogue_id, seeker_t - 1), np.nan)

        delta_E_t = (E_t - E_prev) if not np.isnan(E_prev) else np.nan

        # Cumulative mean of E for all PRIOR seeker turns in same dialogue
        prior_E_values = []
        for lag_i in range(seeker_t):  # 0 .. seeker_t-1
            E_val = train_E_map.get((dialogue_id, lag_i), np.nan)
            if np.isnan(E_val) and test_E_maps and mode == "test":
                route_fold = dialogue_to_fold.get(dialogue_id, 0)
                E_val = test_E_maps.get(route_fold, {}).get((dialogue_id, lag_i), np.nan)
            if not np.isnan(E_val):
                prior_E_values.append(E_val)
        mean_E_lt = np.mean(prior_E_values) if prior_E_values else np.nan

        # --- Uncertainty & context signals ---
        entropy_t = compute_entropy(current_logits)
        margin_t = compute_margin(current_logits)
        probs_t = softmax(current_logits)
        p_c7_minus_c6 = float(probs_t[7] - probs_t[6])

        # Text-based features
        if sample_id in samples:
            sample_dict = samples[sample_id]
            seeker_word_count = len(sample_dict['current_text'].split())
            prev_supp_wc, prev_supp_has_q = get_prev_supporter_info(sample_dict)
        else:
            seeker_word_count = 0
            prev_supp_wc, prev_supp_has_q = 0, 0

        # --- Build feature row ---
        feat_row = {'id': sample_id, 'label': label, 'fold_id': fold_id, 'dialogue_id': dialogue_id}

        # Current logits
        for c in range(9):
            feat_row[f'logit_c{c}'] = float(current_logits[c])

        # Historical logits (27 features)
        for lag_idx, lag in enumerate([1, 2, 3]):
            h_logits = hist_logits[lag_idx]
            for c in range(9):
                feat_row[f'logit_lag{lag}_c{c}'] = float(h_logits[c]) if not np.isnan(h_logits[c]) else np.nan

        # Temporal features
        feat_row['abs_turn_idx'] = abs_t
        feat_row['seeker_turn_idx'] = seeker_t
        feat_row['turn_gap'] = turn_gap

        # DMRS Maturity
        feat_row['E_t'] = E_t
        feat_row['delta_E_t'] = delta_E_t
        feat_row['mean_E_lt'] = mean_E_lt

        # Uncertainty signals
        feat_row['entropy_t'] = entropy_t
        feat_row['margin_t'] = margin_t
        feat_row['p_c7_minus_c6'] = p_c7_minus_c6

        # Context signals
        feat_row['seeker_word_count'] = seeker_word_count
        feat_row['prev_supporter_word_count'] = prev_supp_wc
        feat_row['prev_supporter_has_question'] = prev_supp_has_q

        rows.append(feat_row)

    result_df = pd.DataFrame(rows)
    return result_df

print("build_trajectory_features() defined.")

In [ ]:
# Build training trajectory features
print("Building training trajectory features...")
train_feats_df = build_trajectory_features(
    fold_df=fold_df_sorted,
    oof_logits_array=oof_logits,
    train_data=train_data,
    test_data=test_data,
    test_logits_by_fold=test_logits_by_fold,
    mode="train",
)
print(f"Train features shape: {train_feats_df.shape}")
print(f"Columns: {train_feats_df.columns.tolist()}")
print(f"\nNaN counts:")
nan_counts = train_feats_df.isna().sum()
print(nan_counts[nan_counts > 0])

In [ ]:
# Verify no leakage: for each sample, its fold's OOF logits were computed
# on held-out fold, so the model never saw sample's label during logit computation
print("\n=== Leakage Check ===")
print("OOF logits at position i correspond to fold_id of sample i.")
print("Historical logits (lag 1,2,3) use OOF logits from DIFFERENT positions")
print("(prior seeker turns in same dialogue).")
print()
print("For sample at seeker_t with fold_id=k:")
print("- Current logits: computed by model trained WITHOUT fold k data ✓")
print("- Lag-1 logits: prior turn (seeker_t-1), could be in ANY fold ✓")
print("  (OOF ensures that logit was computed without that position's label)")
print()
print("PASS: No leakage possible in OOF-based trajectory construction.")

# Additional check: ensure NaN pattern is correct (first seeker turn has NaN lags)
first_turns = train_feats_df[train_feats_df['seeker_turn_idx'] == 0]
print(f"\nFirst seeker turns (seeker_turn_idx=0): {len(first_turns)}")
print(f"All lag-1 features are NaN for first turns: {first_turns['logit_lag1_c0'].isna().all()}")
print(f"All lag-2 features are NaN for first turns: {first_turns['logit_lag2_c0'].isna().all()}")
print(f"All lag-3 features are NaN for first turns: {first_turns['logit_lag3_c0'].isna().all()}")

## Section 4 — Model Training (5-Fold CV)

Three sequential classifiers trained under the same 5-fold CV scheme as Stage 1:
1. **LightGBM** — gradient boosting on decision trees
2. **CatBoost** — gradient boosting with ordered boosting
3. **Logistic Regression (L2)** — linear baseline

> **NaN handling:** For each fold, NaN trajectory features (from first seeker turns or cross-split boundary) are imputed with the **column median computed on the training fold only**. This prevents any leakage from the validation set.

In [ ]:
# Define feature columns (exclude metadata)
META_COLS = ['id', 'label', 'fold_id', 'dialogue_id']
FEATURE_COLS = [c for c in train_feats_df.columns if c not in META_COLS]

print(f"Total features: {len(FEATURE_COLS)}")
print("\nFeature groups:")
logit_feats = [c for c in FEATURE_COLS if c.startswith('logit_c')]
hist_feats = [c for c in FEATURE_COLS if 'lag' in c]
temporal_feats = ['abs_turn_idx', 'seeker_turn_idx', 'turn_gap']
maturity_feats = ['E_t', 'delta_E_t', 'mean_E_lt']
uncertainty_feats = ['entropy_t', 'margin_t', 'p_c7_minus_c6']
context_feats = ['seeker_word_count', 'prev_supporter_word_count', 'prev_supporter_has_question']

print(f"  Current logits: {len(logit_feats)}")
print(f"  Historical logits (lag 1-3): {len(hist_feats)}")
print(f"  Temporal: {temporal_feats}")
print(f"  DMRS Maturity: {maturity_feats}")
print(f"  Uncertainty: {uncertainty_feats}")
print(f"  Context: {context_feats}")
print(f"  Total: {len(FEATURE_COLS)}")

In [ ]:
def predict_to_logits(model, X, model_type):
    """Get log-probability (pseudo-logit) predictions from a classifier."""
    if model_type == 'lgbm':
        # LightGBM predict_proba gives probabilities
        proba = model.predict_proba(X)
        return np.log(proba + 1e-10)   # log-probabilities
    elif model_type == 'catboost':
        proba = model.predict_proba(X)
        return np.log(proba + 1e-10)
    elif model_type == 'logreg':
        # LogisticRegression has decision_function (raw scores)
        return model.decision_function(X)  # shape [N, 9]
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

def impute_with_train_median(X_train, X_val, feature_cols):
    """Impute NaN in val using median computed from train only."""
    medians = np.nanmedian(X_train, axis=0)
    X_train_imp = X_train.copy()
    X_val_imp = X_val.copy()
    for j in range(X_train.shape[1]):
        nan_mask_train = np.isnan(X_train_imp[:, j])
        nan_mask_val = np.isnan(X_val_imp[:, j])
        X_train_imp[nan_mask_train, j] = medians[j]
        X_val_imp[nan_mask_val, j] = medians[j]
    return X_train_imp, X_val_imp, medians

print("Utility functions defined.")

In [ ]:
# Initialize OOF logit arrays
N_TRAIN = len(train_feats_df)
N_CLASSES = 9

oof_logits_lgbm = np.zeros((N_TRAIN, N_CLASSES), dtype=np.float32)
oof_logits_catboost = np.zeros((N_TRAIN, N_CLASSES), dtype=np.float32)
oof_logits_logreg = np.zeros((N_TRAIN, N_CLASSES), dtype=np.float32)

# Store trained models for later test inference
fold_models_lgbm = {}
fold_models_catboost = {}
fold_models_logreg = {}
fold_medians = {}  # For imputation at test time

print(f"Training on {N_TRAIN} samples with {len(FEATURE_COLS)} features...")

In [ ]:
# 5-fold training loop
from sklearn.metrics import f1_score

for fold_k in range(cfg.N_FOLDS):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_k}")
    print(f"{'='*60}")
    
    train_mask = train_feats_df['fold_id'] != fold_k
    val_mask = train_feats_df['fold_id'] == fold_k
    
    X_train = train_feats_df.loc[train_mask, FEATURE_COLS].values.astype(np.float32)
    y_train = train_feats_df.loc[train_mask, 'label'].values.astype(int)
    X_val = train_feats_df.loc[val_mask, FEATURE_COLS].values.astype(np.float32)
    y_val = train_feats_df.loc[val_mask, 'label'].values.astype(int)
    val_indices = train_feats_df[val_mask].index.tolist()
    
    print(f"Train: {len(X_train)} samples, Val: {len(X_val)} samples")
    
    # Impute NaN with training fold median
    X_train_imp, X_val_imp, medians = impute_with_train_median(X_train, X_val, FEATURE_COLS)
    fold_medians[fold_k] = medians
    
    print(f"NaN imputed with train median ({np.isnan(X_train).sum()} train NaNs, {np.isnan(X_val).sum()} val NaNs)")
    
    # ===== LightGBM =====
    if HAS_LGBM:
        print("\n--- LightGBM ---")
        lgbm_model = lgb.LGBMClassifier(
            num_leaves=31,
            max_depth=3,
            min_child_samples=25,
            learning_rate=0.05,
            n_estimators=500,
            reg_alpha=0.5,
            reg_lambda=1.0,
            class_weight='balanced',
            random_state=42,
            verbose=-1,
        )
        
        # Callback for early stopping based on macro-f1
        # LightGBM early stopping on custom metric
        def lgb_macro_f1(y_true, y_pred):
            """Custom LGB eval metric: macro F1 on classes 1-8."""
            if hasattr(y_true, 'get_label'):
                labels = y_true.get_label().astype(int)
                pred_proba = y_pred.reshape(9, len(labels)).T
            else:
                labels = y_true.astype(int)
                if y_pred.ndim == 1:
                    pred_proba = y_pred.reshape(len(labels), -1)
                else:
                    pred_proba = y_pred
            preds = np.argmax(pred_proba, axis=1)
            f1 = f1_score(labels, preds, average='macro', labels=list(range(1, 9)), zero_division=0)
            return 'macro_f1_1_8', f1, True
        
        lgbm_model.fit(
            X_train_imp, y_train,
            eval_set=[(X_val_imp, y_val)],
            eval_metric=lgb_macro_f1,
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False),
                lgb.log_evaluation(period=100),
            ],
        )
        
        lgb_preds_proba = lgbm_model.predict_proba(X_val_imp)
        lgb_preds = np.argmax(lgb_preds_proba, axis=1)
        lgb_f1 = f1_score(y_val, lgb_preds, average='macro', labels=list(range(1, 9)), zero_division=0)
        print(f"LightGBM Val Macro-F1 (1-8): {lgb_f1:.4f}")
        print(f"Best iteration: {lgbm_model.best_iteration_}")
        
        lgb_logits = predict_to_logits(lgbm_model, X_val_imp, 'lgbm')
        oof_logits_lgbm[val_indices] = lgb_logits.astype(np.float32)
        fold_models_lgbm[fold_k] = lgbm_model
    else:
        print("LightGBM not available, skipping.")
    
    # ===== CatBoost =====
    if HAS_CATBOOST:
        print("\n--- CatBoost ---")
        # Compute class weights for balanced training
        from collections import Counter
        label_counts = Counter(y_train.tolist())
        total = len(y_train)
        class_weights_cb = []
        for c in range(9):
            cnt = label_counts.get(c, 1)
            class_weights_cb.append(total / (9 * cnt))
        
        cb_model = CatBoostClassifier(
            depth=3,
            min_data_in_leaf=20,
            learning_rate=0.05,
            iterations=500,
            l2_leaf_reg=3,
            class_weights=class_weights_cb,
            random_seed=42,
            eval_metric='TotalF1',
            verbose=False,
            early_stopping_rounds=30,
        )
        
        train_pool = Pool(X_train_imp, label=y_train)
        val_pool = Pool(X_val_imp, label=y_val)
        
        cb_model.fit(
            train_pool,
            eval_set=val_pool,
            verbose=100,
        )
        
        cb_preds_proba = cb_model.predict_proba(X_val_imp)
        cb_preds = np.argmax(cb_preds_proba, axis=1)
        cb_f1 = f1_score(y_val, cb_preds, average='macro', labels=list(range(1, 9)), zero_division=0)
        print(f"CatBoost Val Macro-F1 (1-8): {cb_f1:.4f}")
        
        cb_logits = predict_to_logits(cb_model, X_val_imp, 'catboost')
        oof_logits_catboost[val_indices] = cb_logits.astype(np.float32)
        fold_models_catboost[fold_k] = cb_model
    else:
        print("CatBoost not available, skipping.")
    
    # ===== Logistic Regression =====
    print("\n--- Logistic Regression ---")
    lr_model = LogisticRegression(
        C=0.5,
        multi_class='multinomial',
        solver='saga',
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        n_jobs=-1,
    )
    lr_model.fit(X_train_imp, y_train)
    
    lr_preds = lr_model.predict(X_val_imp)
    lr_f1 = f1_score(y_val, lr_preds, average='macro', labels=list(range(1, 9)), zero_division=0)
    print(f"LogReg Val Macro-F1 (1-8): {lr_f1:.4f}")
    
    lr_logits = predict_to_logits(lr_model, X_val_imp, 'logreg')
    oof_logits_logreg[val_indices] = lr_logits.astype(np.float32)
    fold_models_logreg[fold_k] = lr_model

print("\n" + "="*60)
print("Training complete!")


In [ ]:
# Save OOF logits
os.makedirs(cfg.STAGE2_DIR, exist_ok=True)

if HAS_LGBM:
    lgbm_oof_path = os.path.join(cfg.STAGE2_DIR, "oof_logits_lgbm.npy")
    np.save(lgbm_oof_path, oof_logits_lgbm)
    print(f"Saved LightGBM OOF logits: {oof_logits_lgbm.shape} -> {lgbm_oof_path}")

if HAS_CATBOOST:
    catboost_oof_path = os.path.join(cfg.STAGE2_DIR, "oof_logits_catboost.npy")
    np.save(catboost_oof_path, oof_logits_catboost)
    print(f"Saved CatBoost OOF logits: {oof_logits_catboost.shape} -> {catboost_oof_path}")

logreg_oof_path = os.path.join(cfg.STAGE2_DIR, "oof_logits_logreg.npy")
np.save(logreg_oof_path, oof_logits_logreg)
print(f"Saved LogReg OOF logits: {oof_logits_logreg.shape} -> {logreg_oof_path}")

## Section 5 — Test Inference (Held-Out Routing)

For test samples, we use **held-out routing**: each test sample's dialogue was seen by exactly one fold (the dialogue appears in training data for some fold). We route each test sample to the fold model that was trained WITHOUT that dialogue, ensuring the same leakage-free setup as during OOF training.

Routing logic:
- If `dialogue_id` appears in training data → use fold assigned to that dialogue in `train_folds.parquet`
- If `dialogue_id` is new (not in training) → use fold 0 (default)

> **Note:** `test_logits_by_fold[fold_k]` contains the Stage 1 predictions for all 472 test samples, as predicted by the model trained on `fold_k`'s training data. For test trajectory features, we use the held-out fold's logits.

In [ ]:
# Build trajectory features for test set
print("Building test trajectory features...")
test_feats_df = build_trajectory_features(
    fold_df=fold_df_sorted,
    oof_logits_array=oof_logits,
    train_data=train_data,
    test_data=test_data,
    test_logits_by_fold=test_logits_by_fold,
    mode="test",
)

print(f"Test features shape: {test_feats_df.shape}")
print(f"NaN counts in test features:")
nan_counts_test = test_feats_df[FEATURE_COLS].isna().sum()
print(nan_counts_test[nan_counts_test > 0])

In [ ]:
# Test inference with held-out routing
N_TEST = len(test_feats_df)
test_logits_lgbm = np.zeros((N_TEST, N_CLASSES), dtype=np.float32)
test_logits_catboost = np.zeros((N_TEST, N_CLASSES), dtype=np.float32)

# Map: test sample index -> fold used for routing
# Routing: use the fold of the dialogue from training data
X_test_all = test_feats_df[FEATURE_COLS].values.astype(np.float32)
test_dialogue_ids = test_feats_df['dialogue_id'].values

# Group test samples by their routing fold
fold_groups = {k: [] for k in range(cfg.N_FOLDS)}
for i, diag_id in enumerate(test_dialogue_ids):
    route_fold = dialogue_to_fold.get(diag_id, 0)  # default fold 0 for unseen dialogues
    fold_groups[route_fold].append(i)

print("Test sample routing distribution:")
for fold_k, indices in fold_groups.items():
    print(f"  Fold {fold_k}: {len(indices)} samples")

# Apply each fold's model to its routed test samples
for fold_k in range(cfg.N_FOLDS):
    indices = fold_groups[fold_k]
    if not indices:
        continue
    
    X_test_fold = X_test_all[indices]
    
    # Impute NaN using this fold's training medians
    medians = fold_medians[fold_k]
    for j in range(X_test_fold.shape[1]):
        nan_mask = np.isnan(X_test_fold[:, j])
        X_test_fold[nan_mask, j] = medians[j]
    
    if HAS_LGBM and fold_k in fold_models_lgbm:
        lgb_proba = fold_models_lgbm[fold_k].predict_proba(X_test_fold)
        test_logits_lgbm[indices] = np.log(lgb_proba + 1e-10).astype(np.float32)
    
    if HAS_CATBOOST and fold_k in fold_models_catboost:
        cb_proba = fold_models_catboost[fold_k].predict_proba(X_test_fold)
        test_logits_catboost[indices] = np.log(cb_proba + 1e-10).astype(np.float32)

print(f"\nTest logits shape (LGBM): {test_logits_lgbm.shape}")
print(f"Test logits shape (CatBoost): {test_logits_catboost.shape}")

In [ ]:
# Save test logits
if HAS_LGBM:
    lgbm_test_path = os.path.join(cfg.STAGE2_DIR, "test_logits_routed_lgbm.npy")
    np.save(lgbm_test_path, test_logits_lgbm)
    print(f"Saved LightGBM test logits: {test_logits_lgbm.shape} -> {lgbm_test_path}")

if HAS_CATBOOST:
    catboost_test_path = os.path.join(cfg.STAGE2_DIR, "test_logits_routed_catboost.npy")
    np.save(catboost_test_path, test_logits_catboost)
    print(f"Saved CatBoost test logits: {test_logits_catboost.shape} -> {catboost_test_path}")

print("\nTest inference complete.")

## Section 6 — RQ1 Evaluation

**Research Question (RQ1):** Does adding trajectory features improve Macro-F1 for overlapping classes C5/C6/C7?

We compare:
1. **Stage1 Baseline**: raw OOF logits from Qwen3-8B fine-tuning
2. **Stage1 + LGBM**: trajectory-enhanced predictions
3. **Stage1 + CatBoost**: trajectory-enhanced predictions  
4. **Stage1 + LogReg**: trajectory-enhanced predictions

Evaluation metric: **Macro-F1 on classes 1–8** (excluding class 0 as it is the "no defense" baseline).

In [ ]:
# Align all OOF logits to fold_df order
# Note: train_feats_df may be in different order than fold_df
# We need to get labels aligned with our OOF predictions

# Get labels aligned with train_feats_df
true_labels = train_feats_df['label'].values.astype(int)
sample_ids = train_feats_df['id'].values

# Get Stage1 OOF logits for the same samples (in same order)
# fold_df_sorted is sorted by [dialogue_id, seeker_turn_idx]
# We need to map sample_ids to their row in oof_logits array

# Build id -> oof_logits row mapping
id_to_oof_row = {row['id']: i for i, row in fold_df_sorted.iterrows()}

stage1_oof_logits_aligned = np.zeros((len(train_feats_df), 9), dtype=np.float32)
for local_i, sid in enumerate(sample_ids):
    global_i = id_to_oof_row.get(sid)
    if global_i is not None:
        stage1_oof_logits_aligned[local_i] = oof_logits[global_i]
    else:
        stage1_oof_logits_aligned[local_i] = 0.0  # fallback

print(f"Aligned Stage1 OOF logits: {stage1_oof_logits_aligned.shape}")
print(f"True labels: {true_labels.shape}")

In [ ]:
# Compute OOF Macro-F1 for all models
EVAL_LABELS = list(range(1, 9))   # Classes 1-8
FOCUS_CLASSES = [5, 6, 7]         # C5, C6, C7

def evaluate_logits(logits, true_labels, label_set=EVAL_LABELS):
    """Compute macro-F1 and per-class F1."""
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(true_labels, preds, average='macro', labels=label_set, zero_division=0)
    per_class_f1 = f1_score(true_labels, preds, average=None, labels=list(range(9)), zero_division=0)
    return macro_f1, per_class_f1, preds

results = {}

# Stage 1 baseline
s1_macro_f1, s1_per_class, s1_preds = evaluate_logits(stage1_oof_logits_aligned, true_labels)
results['Stage1 (Qwen3-8B)'] = {
    'macro_f1_1_8': s1_macro_f1,
    **{f'f1_c{c}': s1_per_class[c] for c in range(9)}
}

# Stage2 models
if HAS_LGBM:
    lgbm_macro_f1, lgbm_per_class, lgbm_preds = evaluate_logits(oof_logits_lgbm, true_labels)
    results['Stage1+LGBM'] = {
        'macro_f1_1_8': lgbm_macro_f1,
        **{f'f1_c{c}': lgbm_per_class[c] for c in range(9)}
    }

if HAS_CATBOOST:
    cb_macro_f1, cb_per_class, cb_preds = evaluate_logits(oof_logits_catboost, true_labels)
    results['Stage1+CatBoost'] = {
        'macro_f1_1_8': cb_macro_f1,
        **{f'f1_c{c}': cb_per_class[c] for c in range(9)}
    }

lr_macro_f1, lr_per_class, lr_preds = evaluate_logits(oof_logits_logreg, true_labels)
results['Stage1+LogReg'] = {
    'macro_f1_1_8': lr_macro_f1,
    **{f'f1_c{c}': lr_per_class[c] for c in range(9)}
}

# Print comparison table
results_df = pd.DataFrame(results).T

print("\n" + "="*70)
print("OOF MACRO-F1 COMPARISON (Classes 1-8)")
print("="*70)
display_cols = ['macro_f1_1_8'] + [f'f1_c{c}' for c in FOCUS_CLASSES]
display_df = results_df[display_cols].copy()
display_df.columns = ['Macro-F1 (1-8)'] + [f'F1-C{c}' for c in FOCUS_CLASSES]
print(display_df.to_string(float_format='{:.4f}'.format))
print()

# Full per-class breakdown
print("\n" + "="*70)
print("PER-CLASS F1 BREAKDOWN")
print("="*70)
class_names = ['C0:NoDef', 'C1:Action', 'C2:MajImg', 'C3:Disav', 'C4:MinImg', 
               'C5:Neurot', 'C6:Obsess', 'C7:HighAdp', 'C8:NMI']
per_class_df = results_df[[f'f1_c{c}' for c in range(9)]].copy()
per_class_df.columns = class_names
print(per_class_df.to_string(float_format='{:.4f}'.format))

In [ ]:
# Focus on RQ1: C5/C6/C7 improvement
print("\n" + "="*70)
print("RQ1 ANSWER — C5/C6/C7 F1 COMPARISON")
print("="*70)
focus_df = results_df[[f'f1_c{c}' for c in FOCUS_CLASSES]].copy()
focus_df.columns = ['C5 (Neurotic)', 'C6 (Obsessional)', 'C7 (High-Adaptive)']

# Add delta from Stage1 baseline
baseline_vals = {f'f1_c{c}': results['Stage1 (Qwen3-8B)'][f'f1_c{c}'] for c in FOCUS_CLASSES}
print(focus_df.to_string(float_format='{:.4f}'.format))

print("\nDelta from Stage1 baseline:")
delta_df = focus_df.copy()
for col_i, c in zip(focus_df.columns, FOCUS_CLASSES):
    delta_df[col_i] = focus_df[col_i] - results['Stage1 (Qwen3-8B)'][f'f1_c{c}']

# Only show non-baseline rows
print(delta_df.loc[delta_df.index != 'Stage1 (Qwen3-8B)'].to_string(float_format='{:+.4f}'.format))

best_model = max(
    [k for k in results.keys() if k != 'Stage1 (Qwen3-8B)'],
    key=lambda k: results[k]['macro_f1_1_8']
)
delta_macro = results[best_model]['macro_f1_1_8'] - results['Stage1 (Qwen3-8B)']['macro_f1_1_8']
print(f"\nBest Stage2 model: {best_model}")
print(f"Macro-F1 improvement: {delta_macro:+.4f}")

In [ ]:
# Confusion matrix heatmaps: Stage1 vs Stage1+LGBM (best)
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)

# Determine best stage2 model
best_stage2_preds = None
best_stage2_name = None
if HAS_LGBM:
    best_stage2_preds = lgbm_preds
    best_stage2_name = 'Stage1+LGBM'
elif HAS_CATBOOST:
    best_stage2_preds = cb_preds
    best_stage2_name = 'Stage1+CatBoost'
else:
    best_stage2_preds = lr_preds
    best_stage2_name = 'Stage1+LogReg'

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

class_labels = [f'C{i}' for i in range(9)]

for ax, preds, title in [
    (axes[0], s1_preds, 'Stage1 (Qwen3-8B)'),
    (axes[1], best_stage2_preds, best_stage2_name)
]:
    cm = confusion_matrix(true_labels, preds, labels=list(range(9)), normalize='true')
    sns.heatmap(
        cm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=class_labels, yticklabels=class_labels,
        ax=ax, vmin=0, vmax=1
    )
    macro_f1_val = f1_score(true_labels, preds, average='macro', labels=list(range(1, 9)), zero_division=0)
    ax.set_title(f'{title}\nMacro-F1 (1-8): {macro_f1_val:.4f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('RQ1: Confusion Matrix Comparison (OOF)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

cm_save_path = os.path.join(cfg.RESULTS_DIR, 'confusion_stage1_vs_stage2.png')
plt.savefig(cm_save_path, bbox_inches='tight', dpi=150)
plt.close()
print(f"Saved confusion matrix plot to: {cm_save_path}")

In [ ]:
# Save results CSV
results_save_path = os.path.join(cfg.RESULTS_DIR, 'rq1_model_comparison.csv')
results_df.to_csv(results_save_path)
print(f"Saved comparison table to: {results_save_path}")

print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
print(display_df.to_string(float_format='{:.4f}'.format))

## Section 7 — Feature Importance

LightGBM's built-in feature importance (by `gain`) shows which trajectory features contribute most to predicting DMRS labels. We expect:
- Current logits (direct Stage 1 output) to dominate
- Historical trajectory features (lag-1 especially) to add value for temporal context
- DMRS Maturity Score and entropy to capture uncertainty patterns

In [ ]:
if HAS_LGBM and fold_models_lgbm:
    # Average feature importance across folds
    importances = np.zeros(len(FEATURE_COLS))
    for fold_k, model in fold_models_lgbm.items():
        fi = model.feature_importances_
        importances += fi
    importances /= len(fold_models_lgbm)
    
    # Sort by importance
    sorted_idx = np.argsort(importances)[::-1]
    top_n = 20
    top_idx = sorted_idx[:top_n]
    top_features = [FEATURE_COLS[i] for i in top_idx]
    top_importances = importances[top_idx]
    
    # Create human-readable labels
    def prettify_feat_name(name):
        if name.startswith('logit_c'):
            c = name.split('logit_c')[1]
            return f'Current Logit C{c}'
        elif 'lag' in name:
            parts = name.split('_')
            lag = parts[1].replace('lag', 'Lag-')
            c = parts[2].replace('c', 'C')
            return f'Hist {lag} {c}'
        else:
            labels_map = {
                'E_t': 'DMRS Maturity (E_t)',
                'delta_E_t': 'DMRS Delta E',
                'mean_E_lt': 'DMRS Cumul Mean E',
                'entropy_t': 'Softmax Entropy',
                'margin_t': 'Prob Margin',
                'p_c7_minus_c6': 'P(C7) - P(C6)',
                'abs_turn_idx': 'Abs Turn Idx',
                'seeker_turn_idx': 'Seeker Turn Idx',
                'turn_gap': 'Turn Gap',
                'seeker_word_count': 'Seeker Word Count',
                'prev_supporter_word_count': 'Prev Supp Word Count',
                'prev_supporter_has_question': 'Prev Supp Has ?',
            }
            return labels_map.get(name, name)
    
    pretty_labels = [prettify_feat_name(f) for f in top_features]
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 7))
    colors = ['#2196F3' if 'Current' in l else 
              '#FF9800' if 'Hist' in l else 
              '#4CAF50' if 'DMRS' in l else 
              '#E91E63' if any(x in l for x in ['Entropy', 'Margin', 'P(C']) else 
              '#9C27B0' 
              for l in pretty_labels]
    
    bars = ax.barh(range(top_n), top_importances[::-1], color=colors[::-1], edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(pretty_labels[::-1], fontsize=10)
    ax.set_xlabel('Average Importance (gain, across 5 folds)', fontsize=11)
    ax.set_title('LightGBM Feature Importance (Top 20)\nAveraged across 5-fold CV', 
                 fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#2196F3', label='Current Logits'),
        Patch(facecolor='#FF9800', label='Historical Logits (Lag 1-3)'),
        Patch(facecolor='#4CAF50', label='DMRS Maturity Features'),
        Patch(facecolor='#E91E63', label='Uncertainty Features'),
        Patch(facecolor='#9C27B0', label='Temporal/Context Features'),
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
    
    plt.tight_layout()
    fi_save_path = os.path.join(cfg.RESULTS_DIR, 'lgbm_feature_importance.png')
    plt.savefig(fi_save_path, bbox_inches='tight', dpi=150)
    plt.close()
    print(f"Saved feature importance plot to: {fi_save_path}")
    
    print("\nTop 20 Features by Importance:")
    fi_df = pd.DataFrame({'feature': top_features, 'importance': top_importances})
    print(fi_df.to_string(index=False, float_format='{:.2f}'.format))
else:
    print("LightGBM not available, skipping feature importance plot.")

In [ ]:
# Final summary of all outputs
print("=" * 70)
print("STAGE 2 ARTIFACTS SUMMARY")
print("=" * 70)

artifact_paths = [
    os.path.join(cfg.STAGE2_DIR, "oof_logits_lgbm.npy"),
    os.path.join(cfg.STAGE2_DIR, "oof_logits_catboost.npy"),
    os.path.join(cfg.STAGE2_DIR, "oof_logits_logreg.npy"),
    os.path.join(cfg.STAGE2_DIR, "test_logits_routed_lgbm.npy"),
    os.path.join(cfg.STAGE2_DIR, "test_logits_routed_catboost.npy"),
    os.path.join(cfg.RESULTS_DIR, "confusion_stage1_vs_stage2.png"),
    os.path.join(cfg.RESULTS_DIR, "lgbm_feature_importance.png"),
    os.path.join(cfg.RESULTS_DIR, "rq1_model_comparison.csv"),
]

for path in artifact_paths:
    exists = os.path.exists(path)
    size_str = f"{os.path.getsize(path):,} bytes" if exists else "NOT FOUND"
    status = "✓" if exists else "✗"
    print(f"{status} {os.path.basename(path)}: {size_str}")

print("\n=" * 70)
print("Stage 2 Sequential Trajectory Classification — COMPLETE")
print("=" * 70)